In [1]:
%load_ext autoreload
%autoreload 2
from multistyleseg.experiments.fundus.ensemble import get_ensemble_model
import torch

ensemble = (
    get_ensemble_model(
        model_choices=["UNET", "SEGFORMER", "SERESNET_UNET"], to_onnx=True
    )
    .cpu()
    .eval()
)


In [2]:
import torch

exemple_inputs = torch.randn(
    1, 3, 1024, 1024
)  # Example input tensor with shape (1, 3, 1024, 1024)
# 1. Define symbolic dimensions
batch = torch.export.Dim("batch", min=1, max=1024)
height = torch.export.Dim("height", min=256, max=4096)
width = torch.export.Dim("width", min=256, max=4096)

# 2. Map dimensions ONLY to the input 'x'
dynamic_shapes = {"x": {0: batch, 2: height, 3: width}}

# 3. Export
# Ensure 'input_names' and 'output_names' are still there for the final ONNX file
torch.onnx.export(
    ensemble,
    (exemple_inputs,),
    "ensemble_segmenter.onnx",
    export_params=True,
    do_constant_folding=True,
    dynamo=False,  # Use dynamic_axes instead of dynamic_shapes
    input_names=["input"],
    output_names=["lesions", "od_mac"],
    # Since dynamo=False, we use dynamic_axes instead of dynamic_shapes
    dynamic_axes={
        "input": {0: "batch", 2: "height", 3: "width"},
        "lesions": {0: "batch", 1: "height", 2: "width"},
        "od_mac": {0: "batch", 1: "height", 2: "width"},
    },
)

You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This 

In [3]:
import onnx

onnx_model = onnx.load("ensemble_segmenter.onnx")
onnx.checker.check_model(onnx_model)

In [4]:
import onnxruntime

# Keep the batch dimension: shape (1, 3, 1024, 1024)
onnx_input = exemple_inputs.detach().cpu().numpy()
print(f"Input shape: {onnx_input.shape}")

ort_session = onnxruntime.InferenceSession(
    "./ensemble_segmenter.onnx", providers=["CudaExecutionProvider"]
)

input_name = ort_session.get_inputs()[0].name
onnxruntime_input = {input_name: onnx_input}

# ONNX Runtime returns a list of outputs
onnxruntime_outputs = ort_session.run(None, onnxruntime_input)


Input shape: (1, 3, 1024, 1024)
*************** EP Error ***************
EP Error Unknown Provider Type: CudaExecutionProvider when using ['CudaExecutionProvider']
Falling back to ['CPUExecutionProvider'] and retrying.
****************************************


Specified provider 'CudaExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
